# LC 211 — Design Add and Search Words Data Structure
**Difficulty:** Medium | **Category:** Tries / Prefix Trees
**Pattern:** Trie + DFS wildcard matching with '.'

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px;">
<strong>Core Insight:</strong> Extend a standard Trie with one
twist: the '.' character in a search pattern matches ANY letter.
Handle it with DFS — branch into every child at that position
and return True if any branch eventually matches.
</div>

## Official Problem Statement

Design a data structure that supports adding new words and finding
if a string matches any previously added string.

Implement the `WordDictionary` class:
- `WordDictionary()` — Initializes the object.
- `void addWord(word)` — Adds `word` to the data structure.
- `bool search(word)` — Returns `true` if there is any string in
  the data structure that matches `word`, or `false` otherwise.
  `word` may contain dots `'.'` where dots can be matched with
  any letter.

**Constraints:**
- `1 <= word.length <= 25`
- `word` in `addWord` consists of lowercase English letters.
- `word` in `search` consists of `'.'` or lowercase English letters.
- At most `3` dots in `search` words.
- At most `10^4` calls to `addWord` and `search`.

## What This Is Actually Asking

Build a word dictionary that supports a simple wildcard search.
A dot `'.'` in the search query means "any single letter here".

So searching `"b.."` would match `"bad"`, `"bat"`, `"bay"` — any
three-letter word starting with 'b'.

This is LC 208's Trie with one extra rule: when you hit a dot
during search, you must try all possible children, not just one.
That one rule turns a simple walk into a DFS.

## Walk Through an Example by Hand

```
addWord("bad")   addWord("dad")   addWord("mad")

Trie after inserts:
  root
   ├── 'b' -> 'a' -> 'd' [end]
   ├── 'd' -> 'a' -> 'd' [end]
   └── 'm' -> 'a' -> 'd' [end]

search(".ad")
  pos=0 char='.'  -> try 'b','d','m' (all children of root)
    branch 'b': pos=1 char='a' -> 'b' has child 'a'? Yes
      pos=2 char='d' -> 'a' has child 'd'? Yes -> is_end=True -> True!

search("b..") 
  pos=0 char='b' -> root has 'b'? Yes
  pos=1 char='.' -> try all children of 'b' node -> only 'a'
    branch 'a': pos=2 char='.' -> try all children of 'a' -> only 'd'
      branch 'd': pos=3 end of word -> is_end=True -> True!

search("b.x")  -> eventually hits 'x' not in children -> False
```

## The Picture

After addWord("bad"), addWord("dad"), addWord("mad"):

```
root
 ├── 'b'
 │    └── 'a'
 │         └── 'd' [is_end=True]  <- "bad"
 ├── 'd'
 │    └── 'a'
 │         └── 'd' [is_end=True]  <- "dad"
 └── 'm'
      └── 'a'
           └── 'd' [is_end=True]  <- "mad"
```

search(".ad") at position 0 (dot):
```
  Try 'b' branch -> .ad matches bad -> True
  (would also try 'd' and 'm' if 'b' failed)
```

The DFS helper signature:
```
def dfs(node, index) -> bool:
    if index == len(word): return node.is_end
    ch = word[index]
    if ch == '.':
        return any(dfs(child, index+1)
                   for child in node.children.values())
    if ch not in node.children: return False
    return dfs(node.children[ch], index+1)
```

## When To Use This Pattern

- When you see **wildcard/pattern matching** over a word dictionary,
  think Trie + DFS.
- When you see **'.' matches any character**, think branch-on-dot.
- When standard Trie walk needs to **fork at uncertain positions**,
  think recursive DFS over children.
- When at most a **few wildcards** are guaranteed (≤3 dots here),
  DFS branching stays bounded.
- When problem says "design" + store + query, think Trie class.

## The Approach

Use the same `TrieNode` (children dict + is_end) from LC 208.
`addWord` is identical to `insert` — walk/create nodes, mark end.

For `search`, write a recursive DFS helper that takes the current
node and the current character index. If the character is a letter,
follow that child (or return False if missing). If it's `'.'`,
recurse into every child and return True if any branch succeeds.

Base case: when index reaches the end of the word, return
`node.is_end` — the path must also end a complete word.

In [ ]:
# No extra imports needed beyond builtins
from typing import Optional  # for type hints in docstrings


In [ ]:
def test_harness(WordDictClass):
    """
    Replay sequences of (op, args, expected) tuples.
    op is a method name; expected is None for addWord.
    """
    sequences = [
        # --- Test 1: LeetCode example ---
        [
            ("addWord", ["bad"],  None),
            ("addWord", ["dad"],  None),
            ("addWord", ["mad"],  None),
            ("search",  ["pad"],  False),
            ("search",  ["bad"],  True),
            ("search",  [".ad"],  True),
            ("search",  ["b.."],  True),
        ],
        # --- Test 2: dot at start, middle, end ---
        [
            ("addWord", ["cat"],  None),
            ("search",  ["..t"],  True),
            ("search",  ["c.t"],  True),
            ("search",  ["ca."],  True),
            ("search",  ["..."],  True),
            ("search",  [".x."],  False),
        ],
        # --- Test 3: word not added, length mismatch ---
        [
            ("addWord", ["run"],  None),
            ("search",  ["ru"],   False),
            ("search",  ["runs"], False),
            ("search",  [".un"],  True),
        ],
        # --- Test 4: single char words ---
        [
            ("addWord", ["a"],    None),
            ("addWord", ["z"],    None),
            ("search",  ["."],    True),
            ("search",  ["b"],    False),
        ],
    ]

    passed = 0
    failed = 0

    for t_idx, ops in enumerate(sequences, 1):
        wd = WordDictClass()
        seq_ok = True
        for op, args, expected in ops:
            result = getattr(wd, op)(*args)
            if expected is not None and result != expected:
                print(
                    f"  FAILED Test {t_idx}: {op}({args}) "
                    f"=> {result}, expected {expected}"
                )
                seq_ok = False
        if seq_ok:
            print(f"  PASSED Test {t_idx}")
            passed += 1
        else:
            failed += 1

    print(f"\nResults: {passed} passed, {failed} failed "
          f"out of {passed + failed} tests")


In [ ]:
class TrieNode:
    """Single node: children map + word-end flag."""
    def __init__(self):
        self.children = {}    # char -> TrieNode
        self.is_end   = False


class WordDictionary:
    """
    Word dictionary supporting '.' wildcard search.

    Methods:
        addWord(word)  — Insert word into trie. O(m)
        search(word)   — Return True if word (with dots) matches
                         any stored word. O(26^d * m) where d=dots.
    """

    def __init__(self):
        self.root = TrieNode()
        print("[DEBUG] WordDictionary initialised")

    def addWord(self, word: str) -> None:
        """
        Standard trie insert: walk/create nodes per char,
        mark is_end at the last node.
        """
        print(f"[DEBUG] addWord('{word}')")
        pass  # TODO: implement

    def search(self, word: str) -> bool:
        """
        DFS search with wildcard '.' support.
        Letter char: follow that one child.
        Dot char:    recurse into ALL children.
        """
        print(f"[DEBUG] search('{word}')")

        def dfs(node, index):
            print(f"  [DEBUG] dfs index={index}")
            pass  # TODO: implement

        return dfs(self.root, 0)


In [ ]:
# Uncomment and run when solution is ready
# test_harness(WordDictionary)


## Complexity

| Approach | Time (search) | Space |
|---|---|---|
| Brute force (list scan) | O(n * m) | O(n * m) |
| Hash set | No wildcard support | O(n * m) |
| **Trie + DFS (optimal)** | **O(26^d * m)** | **O(n * m)** |

- m = word length, n = number of stored words
- d = number of dots in search pattern
- Problem guarantees d ≤ 3, so branching stays bounded at 26^3
- `addWord` is always O(m)

## Real World Connection

At Citi, AWS CloudWatch metric names follow structured patterns
like `citi.payments.us.latency`. An ops team might query
`citi.payments.*.latency` — a wildcard for any region. A Trie
with dot-wildcard DFS resolves this without scanning all 6,000
metric names.

Lambda routing rules in a serverless ETL pipeline may include
wildcard patterns to match topic names. A WordDictionary-style
structure lets new rules be added at runtime (`addWord`) and
routes resolved instantly (`search`).

When Prophet ML jobs log results to S3 paths that include date
stamps, a wildcard search like `forecast.2026.03.*.result`
retrieves all results across days — the same branching DFS
pattern applied to S3 prefix enumeration.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra